In [2]:
%load_ext autoreload
%autoreload 2
import os, sys

import re
import numpy as np
import pandas as pd


from inDecay import PATH,my_utils
os.chdir(PATH.main_dir)
from inDecay.models import Topk_Event_Overlapping

In [3]:

import pickle as pkl
from scipy.stats import pearsonr

pj = os.path.join

In [1]:
from qrguide import analysis_fn, transformation

In [4]:
## read Test set 
experiments =["ST_June_2017_BOB_LV7A_DPI7", "ST_June_2017_BOB_LV7B_DPI7" ,
      "ST_June_2017_CHO_LV7A_DPI7", "ST_June_2017_CHO_LV7B_DPI7",
      "ST_June_2017_E14TG2A_LV7A_DPI7", "ST_June_2017_E14TG2A_LV7B_DPI7",
      "ST_June_2017_HAP1_LV7A_DPI7", "ST_June_2017_HAP1_LV7B_DPI7",
      "ST_June_2017_K562_800x_LV7A_DPI7", "ST_June_2017_K562_800x_LV7B_DPI7",]

celltypes = [exp.split("_")[3] for exp in experiments if 'LV7A' in exp]
celltype_rename = ['iPSC', 'CHO', 'mESC', 'HAP1', 'K562']

In [9]:
celltypes, celltype_rename

(['BOB', 'CHO', 'E14TG2A', 'HAP1', 'K562'],
 ['iPSC', 'CHO', 'mESC', 'HAP1', 'K562'])

In [5]:
def read_pkl(path):
    with open(path, 'rb') as f:
        Y = pkl.load(f)
    f.close()
    return Y

def evalute_fn(Y_true_path, Y_pred_path):
    Y_pred =read_pkl(Y_pred_path)
    Y = read_pkl(Y_true_path)

    eval_json = analysis_fn.assessment_recipe_forecast(Y, Y_pred)
    eval_json.update(analysis_fn.assessment_recipe_IDL_forecast(Y, Y_pred))

    eval_df = pd.json_normalize(eval_json)
    return eval_df

# Feat v3

In [25]:
Y_true_path_dict = {
    "iPSC":"pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_BOB_LV7A_DPI7/ForeCast_TestY.pkl",
    "CHO" :"pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_CHO_LV7A_DPI7/ForeCast_TestY.pkl",
    "mESC":"pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_E14TG2A_LV7A_DPI7/ForeCast_TestY.pkl",
    "HAP1":"pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_HAP1_LV7A_DPI7/ForeCast_TestY.pkl",
    "K562":"pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_K562_LV7A_DPI7/ForeCast_TestY.pkl"
}

v3_pred_path_dict = {
    "iPSC": "pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_BOB_LV7A_DPI7/lightning_logs/version_2/checkpoints/epoch=96-val_cre=2.95276666TestPred.pkl",
    "CHO": "pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_CHO_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=54-val_cre=3.28731441TestPred.pkl",
    "mESC": "pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_E14TG2A_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=62-val_cre=3.04122829TestPred.pkl",
    "HAP1": "pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_HAP1_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=53-val_cre=3.25698686TestPred.pkl",
    "K562": "pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_K562_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=90-val_cre=3.35240030TestPred.pkl"
}


In [26]:
v3_df = []
for cell in celltype_rename:
    df = evalute_fn(Y_true_path_dict[cell], v3_pred_path_dict[cell])
    df['celltype'] = cell
    df['model'] = 'featv3'

    v3_df.append(df)

In [29]:
# merge all celltype
featv3_df = pd.concat(v3_df, axis=0).reset_index().iloc[:, 1:]

# rename model as set
featv3_df.rename({"model":"set"}, axis=1, inplace=True)

# feat v4 

## interaction

In [9]:
Y_true_path_dict = {
    "iPSC":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_BOB_LV7A_DPI7/ForeCast_TestY.pkl",
    "CHO" :"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_CHO_LV7A_DPI7/ForeCast_TestY.pkl",
    "mESC":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_E14TG2A_LV7A_DPI7/ForeCast_TestY.pkl",
    "HAP1":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_HAP1_LV7A_DPI7/ForeCast_TestY.pkl",
    "K562":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_K562_LV7A_DPI7/ForeCast_TestY.pkl"
}

Y_pred_path_dict = {
    "iPSC": "pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_BOB_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=77-val_cre=2.94685531TestPred.pkl",
    "CHO":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_CHO_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=66-val_cre=3.27570724TestPred.pkl",
    "mESC":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_E14TG2A_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=93-val_cre=3.03215241TestPred.pkl",
    "HAP1":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_HAP1_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=50-val_cre=3.24646974TestPred.pkl",
    "K562":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_K562_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=84-val_cre=3.34424281TestPred.pkl"
}


In [11]:
interaction_df = []
for cell in celltype_rename:
    df = evalute_fn(Y_true_path_dict[cell], Y_pred_path_dict[cell])
    df['celltype'] = cell
    df['model'] = 'Interaction'

    interaction_df.append(df)

## identity

In [10]:
Y_pred_ident_dict = {
    "iPSC": "pl_trainer_log/ST_featv4_ST_DeepDecay_identity/ST_June_2017_BOB_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=99-val_cre=2.95777583TestPred.pkl",
    "CHO":"pl_trainer_log/ST_featv4_ST_DeepDecay_identity/ST_June_2017_CHO_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=95-val_cre=3.28872061TestPred.pkl",
    "mESC":"pl_trainer_log/ST_featv4_ST_DeepDecay_identity/ST_June_2017_E14TG2A_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=95-val_cre=3.04344153TestPred.pkl",
    "HAP1":"pl_trainer_log/ST_featv4_ST_DeepDecay_identity/ST_June_2017_HAP1_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=98-val_cre=3.25877213TestPred.pkl",
    "K562":"pl_trainer_log/ST_featv4_ST_DeepDecay_identity/ST_June_2017_K562_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=97-val_cre=3.35827565TestPred.pkl"
}

In [12]:
for cell in celltype_rename:
    df = evalute_fn(Y_true_path_dict[cell], Y_pred_ident_dict[cell])
    df['celltype'] = cell
    df['model'] = 'identity'

    interaction_df.append(df)

In [34]:
# merge all celltype
featv4_df = pd.concat(interaction_df, axis=0).reset_index().iloc[:, 1:]

# rename model as set
featv4_df.rename({"model":"set"}, axis=1, inplace=True)
featv4_df['set'] = featv4_df.set.map(
    {"Interaction":"featv4 interaction",
    "identity":"featv4 identity"}
)

# save feat v3 and v4

In [37]:
newfeat_perform = featv3_df.append(featv4_df)
newfeat_perform.to_csv("results/featv3_v4_perform_Aug14.csv", index=False)

/tmp/ipykernel_8470/3940577413.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  newfeat_perform = featv3_df.append(featv4_df)


# merge with previous results

In [42]:
newfeat_df_melt = pd.melt(newfeat_perform, id_vars= ['celltype', 'set'], var_name='metric')

In [22]:
benmar_results = pd.read_csv("results/benchmarking/Benchmarking_result_Jun17.csv")

In [43]:
benmar_results.append(newfeat_df_melt)

/tmp/ipykernel_8470/907181862.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  benmar_results.append(newfeat_df_melt)


,celltype,metric,value,set
0,iPSC,KL divergence,2.378639,FORECasT
1,CHO,KL divergence,1.516647,FORECasT
2,mESC,KL divergence,1.712923,FORECasT
3,HAP1,KL divergence,1.631319,FORECasT
4,K562,KL divergence,1.626391,FORECasT
...,...,...,...,...
175,iPSC,Kendall_tau_IDL,0.667801,featv4 identity
176,CHO,Kendall_tau_IDL,0.759736,featv4 identity
177,mESC,Kendall_tau_IDL,0.740424,featv4 identity
178,HAP1,Kendall_tau_IDL,0.755602,featv4 identity
